# ollama installer

In [1]:
# # 1. Force install zstd and Ollama
# !apt-get install -y zstd
# !curl -fsSL https://ollama.com/install.sh | sh

# import subprocess
# import os
# import time

# # 2. Verify installation and start server
# if os.path.exists('/usr/local/bin/ollama'):
#     # Ensure Ollama server is running in the background using nohup for persistence
#     !nohup ollama serve &

#     print("Ollama server starting...")
#     time.sleep(10) # Give it time to warm up

#     # 3. Pull the model
#     !ollama pull qwen2.5:7b-instruct
# else:
#     print("Ollama installation failed. Check the output above for errors.")

In [2]:
# !pip install ollama
!pip install faster-whisper sounddevice scipy
!pip install edge-tts

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 59.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.2/41.2 MB 18.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.0/39.0 MB 19.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.1/17.1 MB 76.5 MB/s eta 0:00:00


In [3]:
!pip install groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 138.3/138.3 kB 15.6 MB/s eta 0:00:00


In [4]:
!pip install PyPDF2

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 16.3 MB/s eta 0:00:00


In [5]:
import json
import re
from typing import List, Optional, Type, Literal
from pydantic import BaseModel, ValidationError, Field
from groq import Groq
import PyPDF2
from collections import defaultdict
import os

# llm.py

In [6]:
from google.colab import userdata

In [7]:
GROQ_API_KEY = userdata.get('GROQ_API_KEY')

client = Groq(api_key=GROQ_API_KEY)


MODEL_PRESETS = {
    "planner": {"temperature": 0.2, "max_tokens": 700},
    "reviewer": {"temperature": 0.0, "max_tokens": 500},
    "question": {"temperature": 0.6, "max_tokens": 200},
    "scoring": {"temperature": 0.1, "max_tokens": 300},
    "feedback": {"temperature": 0.3, "max_tokens": 700},
}


def generate_with_mode(
    mode: str,
    system_prompt: str,
    user_prompt: str,
    schema_model: Type[BaseModel] | None = None,
    retries: int = 2,
):

    config = MODEL_PRESETS.get(mode)
    if not config:
        raise ValueError(f"Unknown mode: {mode}")

    for attempt in range(retries):

        response = client.chat.completions.create(
            model="llama-3.1-8b-instant",
            temperature=config["temperature"],
            max_tokens=config["max_tokens"],
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt},
            ],
            response_format={"type": "json_object"} if schema_model else None
        )

        content = response.choices[0].message.content.strip()

        # Plain text mode
        if schema_model is None:
            return content

        try:
            raw = json.loads(content)
            validated = schema_model.model_validate(raw)
            return validated.model_dump()

        except Exception as e:

            # 🛠 Automatic Repair Prompt
            repair_prompt = f"""
            The following JSON does NOT match the required schema.

            Required schema:
            {schema_model.model_json_schema()}

            Broken JSON:
            {content}

            Fix the JSON so that it strictly matches the schema.
            Return JSON only.
            """

            repair_response = client.chat.completions.create(
                model="llama-3.1-8b-instant",
                temperature=0.0,
                max_tokens=config["max_tokens"],
                messages=[
                    {"role": "system", "content": "You are a JSON repair assistant."},
                    {"role": "user", "content": repair_prompt},
                ],
                response_format={"type": "json_object"}
            )

            try:
                repaired = json.loads(repair_response.choices[0].message.content)
                validated = schema_model.model_validate(repaired)
                return validated.model_dump()
            except:
                continue

    raise RuntimeError("Model failed to produce valid structured output after retries.")

# state.py

In [8]:
class InterviewState:
    def __init__(self, resume, jd, gap, plan):
        self.resume = resume
        self.jd = jd
        self.gap = gap
        self.plan = plan



# file_loader.py

In [9]:
def load_text_file(path):
    with open(path, "r", encoding="utf-8") as f:
        return f.read()

def load_pdf_file(path):
    text = ""
    with open(path, "rb") as f:
        reader = PyPDF2.PdfReader(f)
        for page in reader.pages:
            text += page.extract_text()
    return text

def load_resume(path):
    if path.endswith(".pdf"):
        return load_pdf_file(path)
    elif path.endswith(".txt"):
        return load_text_file(path)
    else:
        raise ValueError("Unsupported file format")

# Resume Parser

In [10]:
class Project(BaseModel):
    name: str
    description: Optional[str] = ""


class ResumeData(BaseModel):
    candidate_name: str
    extracted_skills: List[str]
    extracted_tools: List[str]
    experience_years: float
    extracted_projects: List[Project]


# print("\nLoading files...")
# resume_path = input("Resume Path: ")
# jd_path = input("JD Path: ")

# resume_text = load_resume(resume_path)
# jd_text = load_resume(jd_path)


def parse_resume(resume_text: str):

    system_prompt = """
    You are an expert technical recruiter.

    Extract structured information from the resume.

    Infer:
    - Technical skills: (languages, frameworks, methodologies, extract skills from projects too)
    - Tools: (software, platforms, cloud services, IDEs)
    - Total years of professional experience: (e.g., 3.5)
    - Projects: (name and description for each project)

    Rules:
    - Avoid duplicates
    - Do not hallucinate
    - Be conservative with experience estimation
    - If information is missing, return empty lists or 0.

    The output must be a JSON object with the following keys:
        {
            "candidate_name": "...",
            "extracted_skills": ["skill1", "skill2"],
            "extracted_tools": ["tool1", "tool2"],
            "experience_years": (float),
            "extracted_projects": [
                {
                    "name": "...",
                    "description": "..."
                }
            ],
        }

    RETURN JSON FORMAT ONLY
    """

    return generate_with_mode(
        mode="planner",
        system_prompt=system_prompt,
        user_prompt=resume_text,
        schema_model=ResumeData   # STRICT JSON SCHEMA
    )


def resume_reviewer_pass(resume_text: str, planner_output: dict):

    review_prompt = f"""
    You are a strict validation system.

    Validate the extracted resume data.

    Rules:
    - Remove hallucinated skills/tools not clearly mentioned
    - Remove duplicates
    - Ensure experience_years is realistic
    - Keep only actual projects mentioned
    - If data is invalid, correct it conservatively.

    RESUME:
    {resume_text}

    EXTRACTED:
    {planner_output}

    RETURN JSON FORMAT ONLY
    """

    return generate_with_mode(
        mode="reviewer",
        system_prompt=review_prompt,
        user_prompt="",
        schema_model=ResumeData   # STRICT AGAIN
    )


def robust_parse_resume(resume_text: str):

    try:
        # Step 1: Strict structured extraction
        planner_output = parse_resume(resume_text)

        # Step 2: Strict structured review
        reviewed_output = resume_reviewer_pass(
            resume_text,
            planner_output
        )

        return reviewed_output

    except Exception as e:
        raise RuntimeError(f"Resume parsing pipeline failed: {e}")


# model_resume = robust_parse_resume(resume_text)

# print("\nParsed Resume Data:")
# for key, value in model_resume.items():
#     print(f"{key}: {value}")

# jd_agent.py

In [11]:
class JDRequirements(BaseModel):
    job_title: str
    extracted_skills: List[str]
    extracted_tools: List[str]
    experience_required: str

def parse_jd(jd_text: str):

    system_prompt = """
    You are an expert HR analyst.

    Extract structured hiring requirements from the job description.

    Infer required skills, tools, and experience even if not explicitly labeled.

    Look for phrases like:
    - "proficient in"
    - "experience with"
    - "must have"
    - "strong knowledge of"
    - "minimum X years"
    - implied technical capabilities

    Rules:
    - Avoid duplicates
    - Do not hallucinate
    - If experience years are unclear, summarize conservatively
    - If data is missing, return empty lists or empty string

    The output must be a JSON object with the following keys:
        {
            "job_title": "...",
            "extracted_skills": ["skill1", "skill2"],
            "extracted_tools": ["tool1", "tool2"],
            "experience_required": (extract a Number or Float if possible),
        }

    RETURN JSON FORMAT ONLY
    """

    return generate_with_mode(
        mode="planner",
        system_prompt=system_prompt,
        user_prompt=jd_text,
        schema_model=JDRequirements  # STRICT STRUCTURE
    )


def reviewer_pass(jd_text: str, planner_output: dict):

    review_prompt = f"""
    You are a strict validation system.

    Validate the extracted job requirements.

    Rules:
    - Remove hallucinated tools not mentioned or clearly implied
    - Remove duplicates
    - Ensure experience aligns with JD
    - Keep only realistic skills/tools
    - If data is incorrect, correct conservatively

    JOB DESCRIPTION:
    {jd_text}

    EXTRACTED:
    {planner_output}

    RETURN JSON FORMAT ONLY
    """

    return generate_with_mode(
        mode="reviewer",
        system_prompt=review_prompt,
        user_prompt="",
        schema_model=JDRequirements  # STRICT AGAIN
    )


def robust_parse_jd(jd_text: str):

    try:
        # Step 1: Strict extraction
        planner_output = parse_jd(jd_text)

        # Step 2: Strict review
        reviewed_output = reviewer_pass(jd_text, planner_output)

        return reviewed_output

    except Exception as e:
        raise RuntimeError(f"JD parsing pipeline failed: {e}")


# model_jd = robust_parse_jd(jd_text)

# print("\nParsed JD Data:")
# for key, value in model_jd.items():
#     print(f"{key}: {value}")

# gap_analysis.py

In [12]:
class GapAnalysis(BaseModel):
    strong_skills: List[str]
    partial_match_skills: List[str]
    skill_gaps: List[str]
    resume_experience_years: float
    experience_required: str
    readiness_score: int = Field(ge=0, le=100)

def analyze_gap(resume_data, jd_data):

    system_prompt = """
    You are a hiring analyst.

    Compare candidate resume skills with job description requirements.

    Identify:
    - strong_skills
    - partial_match_skills
    - skill_gaps
    - resume_experience_years
    - experience_required
    - readiness_score (0-100 integer)

    SCORING LOGIC:
    - Strong alignment → higher score
    - Missing core skills → reduce score
    - Experience mismatch → reduce score
    - Be realistic and conservative

    RULES:
    - Do not hallucinate
    - If unsure, classify conservatively
    - Return structured output only

    RETURN JSON FORMAT ONLY
    """

    user_prompt = f"""
    Resume Skills: {resume_data.get("extracted_skills", [])}
    Resume Tools: {resume_data.get("extracted_tools", [])}
    Resume Experience Years: {resume_data.get("experience_years", 0)}

    JD Required Skills: {jd_data.get("extracted_skills", [])}
    JD Required Tools: {jd_data.get("extracted_tools", [])}
    JD Experience Required: {jd_data.get("experience_required", "")}
    """

    gap_analysis = generate_with_mode(
        mode="planner",
        system_prompt=system_prompt,
        user_prompt=user_prompt,
        schema_model=GapAnalysis   # STRICT STRUCTURE
    )

    print("\nGap Analysis:")
    for key, value in gap_analysis.items():
        print(f"\n{key}: {value}")

    return gap_analysis

# gap_data = analyze_gap(model_resume, model_jd)

# planner_agent.py

In [13]:
class InterviewRound(BaseModel):
    type: Literal["Technical", "Behavioral", "Case Study", "Situational", "Domain-Specific", "Warm-up"]
    focus_areas: List[str]
    stress_test_areas: List[str]


class InterviewPlan(BaseModel):
    difficulty: Literal["BEGINNER", "INTERMEDIATE", "ADVANCED"]
    rounds: List[InterviewRound]


def _safe_list(items, fallback):
    cleaned = [str(x).strip() for x in (items or []) if str(x).strip()]
    return cleaned if cleaned else fallback


def _fallback_interview_plan(gap_data, experience_years):
    readiness_score = gap_data.get("readiness_score", 0)
    strong_skills = _safe_list(gap_data.get("strong_skills", []), ["Core Role Fundamentals"])
    partial_skills = _safe_list(gap_data.get("partial_match_skills", []), ["Communication and Collaboration"])
    skill_gaps = _safe_list(gap_data.get("skill_gaps", []), ["Role-Specific Problem Solving"])

    if readiness_score < 40:
        difficulty = "BEGINNER"
    elif readiness_score <= 70:
        difficulty = "INTERMEDIATE"
    else:
        difficulty = "ADVANCED"

    rounds = [
        InterviewRound(
            type="Warm-up",
            focus_areas=["Candidate Introduction", "Experience Overview", "Career Aspirations"],
            stress_test_areas=[]
        ),
        InterviewRound(
            type="Domain-Specific",
            focus_areas=strong_skills[:2],
            stress_test_areas=skill_gaps[:2],
        ),
        InterviewRound(
            type="Behavioral",
            focus_areas=partial_skills[:2],
            stress_test_areas=["Communication", "Decision Making"],
        ),
        InterviewRound(
            type="Situational",
            focus_areas=skill_gaps[:2],
            stress_test_areas=["Trade-offs", "Risk Management"],
        ),
    ]

    return InterviewPlan(difficulty=difficulty, rounds=rounds)


def generate_interview_plan(gap_data, experience_years):

    strong_skills = gap_data.get("strong_skills", [])
    partial_skills = gap_data.get("partial_match_skills", [])
    skill_gaps = gap_data.get("skill_gaps", [])
    readiness_score = gap_data.get("readiness_score", 0)

    system_prompt = """
    You are a universal professional interviewer.

    You conduct structured, role-specific interviews across ANY domain (engineering, data, product, design, marketing, sales, operations, finance, HR, support, healthcare, legal, etc.).

    Create a detailed interview progression plan.

    Responsibilities:
    - Adapt interview type based on readiness score
    - Adjust difficulty based on experience
    - Include appropriate round types for the role context
    - Probe skill gaps intelligently

    Scoring Rules:
    - readiness < 40 → BEGINNER
    - readiness 40–70 → INTERMEDIATE
    - readiness > 70 → ADVANCED

    Strict output rules:
    - Return valid JSON only.
    - difficulty must be one of: BEGINNER, INTERMEDIATE, ADVANCED
    - rounds must be a non-empty list.
    - each round type must be one of: Technical, Behavioral, Case Study, Situational, Domain-Specific
    - each round must include at least one focus_areas item.
    - stress_test_areas can be empty but must be a list.
    - Do not assume software/backend context unless the inputs explicitly indicate it.

    RETURN JSON FORMAT ONLY
    """

    user_prompt = f"""
    Strong skills: {strong_skills}
    Partial skills: {partial_skills}
    Skill gaps: {skill_gaps}
    Readiness score: {readiness_score}
    Experience years: {experience_years}
    """

    try:
        interview_plan_raw = generate_with_mode(
            mode="planner",
            system_prompt=system_prompt,
            user_prompt=user_prompt,
            schema_model=InterviewPlan   # STRICT STRUCTURE
        )

        rounds_raw = interview_plan_raw.get("rounds", []) or []
        validated_rounds = []
        for rnd in rounds_raw:
            round_type = rnd.get("type") if isinstance(rnd, dict) else None
            if round_type not in ["Technical", "Behavioral", "Case Study", "Situational", "Domain-Specific"]:
                continue
            validated_rounds.append(
                InterviewRound(
                    type=round_type,
                    focus_areas=_safe_list(rnd.get("focus_areas", []), ["Core Role Fundamentals"]),
                    stress_test_areas=_safe_list(rnd.get("stress_test_areas", []), []),
                )
            )

        if not validated_rounds:
            raise ValueError("No valid rounds produced by model")

        warmup_round = InterviewRound(
            type="Warm-up",
            focus_areas=["Candidate Introduction", "Experience Overview", "Career Aspirations"],
            stress_test_areas=[]
        )

        difficulty = interview_plan_raw.get("difficulty", "BEGINNER")
        if difficulty not in ["BEGINNER", "INTERMEDIATE", "ADVANCED"]:
            difficulty = "BEGINNER"

        interview_plan = InterviewPlan(
            difficulty=difficulty,
            rounds=[warmup_round] + validated_rounds
        )

    except Exception as e:
        print(f"Planner output invalid, using fallback plan. Reason: {e}")
        interview_plan = _fallback_interview_plan(gap_data, experience_years)

    print("\nInterview Plan:")
    for key, value in interview_plan.model_dump().items():
        print(f"\n{key}: {value}")

    return interview_plan

# intterview_conductor.py

In [14]:
class AnswerEvaluation(BaseModel):
    depth: int = Field(ge=0, le=100)
    clarity: int = Field(ge=0, le=100)
    confidence: int = Field(ge=0, le=100)
    speech_clarity: int = Field(ge=0, le=100)
    needs_followup: bool
    followup_type: Literal["depth_probe", "clarification", "tradeoff"]


def _normalize_text(text: str) -> str:
    lowered = (text or "").lower().strip()
    lowered = re.sub(r"[^a-z0-9\s]", " ", lowered)
    lowered = re.sub(r"\s+", " ", lowered).strip()
    return lowered


def _sanitize_answer_text(answer: str) -> str:
    text = (answer or "").strip()
    if not text:
        return ""

    transcript_markers = [
        r"\buser\s*:",
        r"\bgithub\s*copilot\s*:",
        r"\bai\s*question\s*:",
        r"\bai\s*follow-up\s*:",
        r"\bcandidate\s*:",
    ]

    if any(re.search(pattern, text, flags=re.IGNORECASE) for pattern in transcript_markers):
        copilot_splits = re.split(r"(?i)\bgithub\s*copilot\s*:", text)
        if len(copilot_splits) > 1:
            text = copilot_splits[-1].strip()

        text = re.sub(r"(?im)^\s*(user|candidate|ai question|ai follow-up)\s*:\s*", "", text)

    text = re.sub(r"```[\s\S]*?```", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def _is_meta_or_assisted_answer(answer: str) -> bool:
    text = (answer or "").lower()
    markers = [
        "github copilot:",
        "user:",
        "ai question:",
        "ai follow-up:",
        "answer this question for me",
        "here's a strong sample response",
        "ai overview",
    ]
    hits = sum(1 for marker in markers if marker in text)
    return hits >= 2


def _contains_security_red_flags(answer: str) -> bool:
    text = (answer or "").lower()
    patterns = [
        r"\b(bypass|disable|turn off|remove)\b.{0,30}\b(auth|authentication|authorization|2fa|mfa|security)\b",
        r"\bstore\b.{0,30}\b(password|credentials?|api key|token|secret)\b.{0,30}\b(plain\s*text|openly|unencrypted)\b",
        r"\bhardcod(e|ing)\b.{0,25}\b(password|credentials?|api key|token|secret)\b",
    ]
    return any(re.search(pattern, text, flags=re.IGNORECASE | re.DOTALL) for pattern in patterns)


def _has_concrete_evidence(answer: str) -> bool:
    text = answer or ""
    patterns = [
        r"\bfor example\b",
        r"\bfor instance\b",
        r"\bin my (last|previous|recent)\b",
        r"\bin a project\b",
        r"\bwe (implemented|built|reduced|improved|launched|delivered)\b",
        r"\bresult(ed)? in\b",
        r"\b\d+\s*(%|ms|sec|seconds|minutes|hours|days|x)\b",
        r"\b(before|after)\b",
    ]
    return any(re.search(p, text, flags=re.IGNORECASE) for p in patterns)


def _vagueness_signals(answer: str) -> dict:
    normalized = _normalize_text(answer)
    word_count = len(normalized.split())

    vague_phrases = [
        "at a high level",
        "in general",
        "it depends",
        "best practices",
        "i would focus on",
        "i would aim to",
        "generally speaking",
        "overall approach",
        "balance learning and recharge",
    ]

    vague_hits = sum(1 for phrase in vague_phrases if phrase in normalized)
    has_concrete = _has_concrete_evidence(answer)

    is_vague = (
        (word_count < 45 and vague_hits >= 1 and not has_concrete)
        or (word_count >= 45 and vague_hits >= 3 and not has_concrete)
    )

    return {
        "word_count": word_count,
        "vague_hits": vague_hits,
        "has_concrete": has_concrete,
        "is_vague": is_vague,
    }


def _question_needs_concrete_example(question: str) -> bool:
    q = _normalize_text(question)
    triggers = [
        "how would you",
        "walk me through",
        "describe a real project",
        "what trade offs",
        "what trade-offs",
        "challenge you faced",
    ]
    return any(trigger in q for trigger in triggers)


def _followup_depth_threshold(difficulty: str, round_type: str) -> int:
    base = 60
    if difficulty == "BEGINNER":
        base = 55
    elif difficulty == "ADVANCED":
        base = 65

    if round_type == "Warm-up":
        base -= 5
    elif round_type in ["Case Study", "Technical", "Domain-Specific"]:
        base += 5

    return max(45, min(75, base))


def _should_force_depth_followup(vagueness: dict, question: str, round_type: str, difficulty: str, evaluation: dict) -> bool:
    if round_type == "Warm-up":
        return False

    depth_threshold = _followup_depth_threshold(difficulty, round_type)
    depth_score = int(evaluation.get("depth", 50))

    lacks_concrete_for_concrete_question = _question_needs_concrete_example(question) and not vagueness["has_concrete"]
    strong_vagueness = vagueness["is_vague"] and vagueness["vague_hits"] >= 2

    return (
        strong_vagueness
        or (lacks_concrete_for_concrete_question and depth_score < max(depth_threshold + 8, 68))
        or (vagueness["is_vague"] and depth_score < depth_threshold)
    )


def _jaccard_similarity(a: str, b: str) -> float:
    a_set = set(_normalize_text(a).split())
    b_set = set(_normalize_text(b).split())
    if not a_set or not b_set:
        return 0.0
    return len(a_set & b_set) / len(a_set | b_set)


def _extract_single_question(raw_text: str, fallback: str) -> str:
    text = (raw_text or "").strip()
    if not text:
        return fallback

    try:
        parsed = json.loads(text)
        if isinstance(parsed, dict):
            if isinstance(parsed.get("question"), str) and parsed["question"].strip():
                text = parsed["question"].strip()
            else:
                for value in parsed.values():
                    if isinstance(value, str) and "?" in value:
                        text = value.strip()
                        break
                    if isinstance(value, list):
                        for item in value:
                            if isinstance(item, str) and "?" in item:
                                text = item.strip()
                                break
        elif isinstance(parsed, list):
            for item in parsed:
                if isinstance(item, str) and "?" in item:
                    text = item.strip()
                    break
    except Exception:
        pass

    lines = [ln.strip() for ln in text.splitlines() if ln.strip()]
    cleaned_lines = []
    for ln in lines:
        candidate = ln.lstrip("-*• ")
        if ". " in candidate and candidate.split(". ", 1)[0].isdigit():
            candidate = candidate.split(". ", 1)[1]
        cleaned_lines.append(candidate)

    selected = ""
    for ln in cleaned_lines:
        if ln.endswith("?"):
            selected = ln
            break
    if not selected:
        for ln in cleaned_lines:
            if "?" in ln:
                selected = ln.split("?")[0].strip() + "?"
                break

    if not selected:
        parts = re.split(r"(?<=[.!?])\s+", text)
        for part in parts:
            candidate = part.strip()
            if "?" in candidate:
                selected = candidate if candidate.endswith("?") else candidate.split("?")[0].strip() + "?"
                break

    selected = (selected or fallback).strip().strip('"').strip("'")
    if not selected.endswith("?"):
        selected = selected.rstrip(".") + "?"
    return selected


def _is_generic_question(question: str, topic: str) -> bool:
    q = _normalize_text(question)
    topic_norm = _normalize_text(topic)
    generic_patterns = [
        f"can you explain your understanding of {topic_norm}",
        "can you explain your understanding",
        "with one practical example",
    ]
    return any(pattern in q for pattern in generic_patterns)


def _is_repetitive_question(question: str, previous_questions: List[str], threshold: float = 0.72) -> bool:
    for previous in previous_questions:
        if _jaccard_similarity(question, previous) >= threshold:
            return True
    return False


def _heuristic_scores(answer: str):
    text = (answer or "").strip()
    word_count = len(text.split())
    sentence_count = max(1, len(re.findall(r"[.!?]", text)))
    unique_ratio = 0.0
    words = [w for w in _normalize_text(text).split() if w]
    if words:
        unique_ratio = len(set(words)) / len(words)

    substance_terms = [
        "strategy", "impact", "result", "outcome", "stakeholder", "priority", "risk", "tradeoff",
        "customer", "process", "quality", "metric", "analysis", "execution", "decision", "improvement",
        "implementation", "plan", "constraint", "learning",
    ]
    substance_hits = sum(1 for term in substance_terms if re.search(rf"\b{re.escape(term)}\b", text, flags=re.IGNORECASE))

    filler_words = ["um", "uh", "like", "you know", "basically", "kind of", "sort of"]
    filler_hits = sum(text.lower().count(filler) for filler in filler_words)

    hedge_words = ["maybe", "perhaps", "i think", "not sure", "probably"]
    confidence_penalty = sum(text.lower().count(word) for word in hedge_words) * 4

    depth = 30 + min(40, substance_hits * 6) + min(25, int(word_count / 6))
    clarity = 35 + min(20, int(unique_ratio * 30)) + min(20, sentence_count * 2)
    confidence = 40 + min(25, int(word_count / 8)) - confidence_penalty
    speech_clarity = 60 - min(30, filler_hits * 4) + min(20, int(sentence_count * 1.5))

    if word_count < 12:
        depth -= 25
        clarity -= 15
        confidence -= 20
        speech_clarity -= 15

    if "ai overview" in text.lower():
        clarity -= 10
        speech_clarity -= 8

    def clamp(value):
        return max(0, min(100, int(round(value))))

    return {
        "depth": clamp(depth),
        "clarity": clamp(clarity),
        "confidence": clamp(confidence),
        "speech_clarity": clamp(speech_clarity),
    }


def _merge_scores(llm_eval: dict, heuristic_eval: dict):
    merged = {}
    for key in ["depth", "clarity", "confidence", "speech_clarity"]:
        llm_value = int(llm_eval.get(key, 50))
        heur_value = int(heuristic_eval.get(key, 50))
        merged[key] = max(0, min(100, int(round(0.75 * llm_value + 0.25 * heur_value))))
    return merged


def generate_question(topic, difficulty, round_type, previous_questions):
    topic_lower = (topic or "").lower()
    is_warmup = round_type == "Warm-up" or "candidate introduction" in topic_lower

    warmup_questions = [
        "Could you please introduce yourself and briefly walk me through your background?",
        "Tell me about your recent experience and the kind of work you have been focusing on.",
        "What role are you currently targeting, and why does it fit your strengths?",
    ]

    if is_warmup:
        fallback = warmup_questions[min(len(previous_questions), len(warmup_questions) - 1)]
        system_prompt = f"""
        You are a universal professional interviewer starting the warm-up stage.

        Ask a natural real-life opening interview question suitable for any role/domain.

        OUTPUT CONTRACT (STRICT):
        - Return EXACTLY ONE question.
        - Return plain text only.
        - Do not ask conceptual/theory questions about introductions.
        - Do not repeat previous questions.
        - Keep it concise and conversational.

        Previous questions to avoid: {previous_questions}
        """
    else:
        fallback_options = [
            f"Can you describe a real project or task where you applied {topic}?",
            f"What trade-offs did you consider while applying {topic} in a real work situation?",
            f"Can you walk me through one challenge you faced with {topic} and how you solved it?",
        ]
        fallback = fallback_options[len(previous_questions) % len(fallback_options)]
        system_prompt = f"""
        You are a universal professional interviewer.

        Conduct a {round_type} interview for ANY domain.

        Guidance by round:
        - Behavioral: ask situational and experience-based STAR-style questions.
        - Technical: ask role-practical application questions (not limited to software).
        - Domain-Specific: ask context-aware questions tied to the candidate's target function.
        - Case Study/Situational: ask decision-making and trade-off questions.

        Difficulty: {difficulty}
        Topic: {topic}

        OUTPUT CONTRACT (STRICT):
        - Return EXACTLY ONE question.
        - Return plain text only.
        - Do not return headings, bullet points, numbering, categories, or multiple questions.
        - Do not ask generic "explain your understanding" prompts.
        - Keep it under 30 words.

        Do NOT repeat previous questions: {previous_questions}
        """

    question = fallback
    for _ in range(3):
        response = generate_with_mode(
            mode="question",
            system_prompt=system_prompt,
            user_prompt=""
        )
        candidate = _extract_single_question(response, fallback)
        if _is_generic_question(candidate, topic) and not is_warmup:
            continue
        if _is_repetitive_question(candidate, previous_questions):
            continue
        question = candidate
        break

    return question


def analyze_answer(topic, difficulty, round_type, question, answer, previous_answers=None):
    previous_answers = previous_answers or []
    sanitized_answer = _sanitize_answer_text(answer)
    scoring_answer = sanitized_answer or (answer or "")

    system_prompt = """
    You are a strict evaluator for a universal, voice-based mock interview.

    Evaluate the candidate answer objectively using a 0-100 scale for ANY role/domain.

    Return ONLY a JSON object with these keys:
    - depth (0-100): role-relevant depth, correctness, judgment, and completeness
    - clarity (0-100): logical structure and readability of the response
    - confidence (0-100): decisiveness and ownership in explanation
    - speech_clarity (0-100): spoken communication quality (fluency, filler control, coherence)
    - needs_followup (true/false)
    - followup_type (depth_probe, clarification, tradeoff)

    Scoring guidance:
    - 0-30: weak/incomplete
    - 31-50: basic but fragmented
    - 51-70: acceptable with gaps
    - 71-85: strong and structured
    - 86-100: excellent, precise, and role-ready

    Follow-up rules:
    - Set needs_followup=true if any score is below 60
    - depth low -> depth_probe
    - clarity or speech_clarity low -> clarification
    - good depth but weak reasoning on choices -> tradeoff

    Be conservative and do not hallucinate.
    RETURN JSON FORMAT ONLY.
    """

    user_prompt = f"""
    Question: {question}
    Topic: {topic}
    Round type: {round_type}
    Difficulty: {difficulty}
    Candidate answer transcript: {scoring_answer}
    """

    evaluation = generate_with_mode(
        mode="scoring",
        system_prompt=system_prompt,
        user_prompt=user_prompt,
        schema_model=AnswerEvaluation
    )

    heuristics = _heuristic_scores(scoring_answer)
    blended = _merge_scores(evaluation, heuristics)
    evaluation.update(blended)

    normalized_previous = [_sanitize_answer_text(a) for a in previous_answers[-3:]]
    for previous_answer in normalized_previous:
        if previous_answer and _jaccard_similarity(scoring_answer, previous_answer) >= 0.82:
            evaluation["depth"] = max(0, evaluation["depth"] - 15)
            evaluation["clarity"] = max(0, evaluation["clarity"] - 10)
            evaluation["confidence"] = max(0, evaluation["confidence"] - 15)
            evaluation["needs_followup"] = True
            evaluation["followup_type"] = "clarification"
            break

    if _is_meta_or_assisted_answer(answer):
        evaluation["depth"] = min(evaluation.get("depth", 50), 45)
        evaluation["clarity"] = min(evaluation.get("clarity", 50), 55)
        evaluation["confidence"] = min(evaluation.get("confidence", 50), 45)
        evaluation["needs_followup"] = True
        evaluation["followup_type"] = "clarification"

    vagueness = _vagueness_signals(scoring_answer)

    if vagueness["is_vague"]:
        penalty_scale = 1.0
        if round_type == "Warm-up":
            penalty_scale = 0.45
        elif difficulty == "BEGINNER":
            penalty_scale = 0.7

        evaluation["depth"] = max(0, int(round(evaluation.get("depth", 50) - (18 * penalty_scale))))
        evaluation["clarity"] = max(0, int(round(evaluation.get("clarity", 50) - (8 * penalty_scale))))
        evaluation["confidence"] = max(0, int(round(evaluation.get("confidence", 50) - (10 * penalty_scale))))

    if _should_force_depth_followup(vagueness, question, round_type, difficulty, evaluation):
        evaluation["needs_followup"] = True
        evaluation["followup_type"] = "depth_probe"

    if _contains_security_red_flags(scoring_answer):
        evaluation["depth"] = min(evaluation.get("depth", 50), 25)
        evaluation["confidence"] = min(evaluation.get("confidence", 50), 25)
        evaluation["clarity"] = min(evaluation.get("clarity", 50), 45)
        evaluation["speech_clarity"] = min(evaluation.get("speech_clarity", 50), 45)
        evaluation["needs_followup"] = True
        evaluation["followup_type"] = "clarification"

    low_metric = min(
        [
            ("depth", evaluation.get("depth", 50)),
            ("clarity", evaluation.get("clarity", 50)),
            ("speech_clarity", evaluation.get("speech_clarity", 50)),
        ],
        key=lambda item: item[1]
    )[0]

    if any(evaluation.get(metric, 100) < 60 for metric in ["depth", "clarity", "confidence", "speech_clarity"]):
        evaluation["needs_followup"] = True
        if low_metric == "depth":
            evaluation["followup_type"] = "depth_probe"
        else:
            evaluation["followup_type"] = "clarification"
    elif evaluation.get("depth", 0) >= 75 and evaluation.get("clarity", 0) < 70:
        evaluation["needs_followup"] = True
        evaluation["followup_type"] = "tradeoff"
    else:
        evaluation["needs_followup"] = False

    print("\nAnswer Evaluation:")
    for key, value in evaluation.items():
        print(f"{key}: {value}\n")

    return evaluation


def generate_followup(question, answer, followup_type):
    system_prompt = f"""
    The candidate's answer was weak.

    Ask one {followup_type} follow-up question.

    OUTPUT CONTRACT (STRICT):
    - Return EXACTLY ONE follow-up question.
    - Return plain text only.
    - Do not return headings, bullet points, numbering, categories, or multiple questions.
    - Keep it under 25 words.
    """

    user_prompt = f"""
    Original question: {question}
    Candidate answer: {answer}
    """

    response = generate_with_mode(
        mode="question",
        system_prompt=system_prompt,
        user_prompt=user_prompt
    )

    fallback_map = {
        "depth_probe": "Can you give one specific example with context, action, and measurable result?",
        "clarification": "Can you answer directly in your own words without quoting external text?",
        "tradeoff": "What trade-offs did you consider, and why did you choose that approach?",
    }
    fallback = fallback_map.get(followup_type, "Can you answer the previous question with one concrete example from your experience?")
    return _extract_single_question(response, fallback)

# interview_runtime.py

In [15]:
def run_interview(plan, voice_mode=False):

    all_scores = []
    questions_per_round = 3

    for round_data in plan.get("rounds", []):

        focus_areas = round_data.get("focus_areas") or [round_data.get("type", "General")]
        topic = focus_areas[0]
        base_difficulty = plan.get("difficulty", "BEGINNER")
        current_difficulty = base_difficulty

        print(f"\n=== Round: {round_data['type']} | Topic: {topic} ===\n")

        previous_questions = []

        for i in range(questions_per_round):

            if len(all_scores) >= 2:
                recent = all_scores[-2:]
                if all(s["depth"] >= 75 for s in recent):
                    current_difficulty = "ADVANCED"
                elif all(s["depth"] <= 45 for s in recent):
                    current_difficulty = "BEGINNER"
                else:
                    current_difficulty = base_difficulty

            question = generate_question(
                topic,
                current_difficulty,
                round_data["type"],
                previous_questions
            )

            print(f"\nAI Question: {question}\n")
            previous_questions.append(question)

            if voice_mode:
                text_to_speech(question)
                audio_file = record_audio()
                answer = speech_to_text(audio_file)
            else:
                answer = input("Your Answer: ")

            attempts = 0
            while (not answer or len(answer.strip()) < 10) and attempts < 2:
                print("Answer too short. Please elaborate.")
                attempts += 1
                answer = input("Your Answer: ")

            print(f"\nCandidate: {answer}\n")

            recent_answers = [s.get("answer_text", "") for s in all_scores[-5:] if s.get("answer_text")]
            analysis = analyze_answer(
                topic,
                current_difficulty,
                round_data["type"],
                question,
                answer,
                previous_answers=recent_answers,
            )

            analysis.update({
                "topic": topic,
                "difficulty": current_difficulty,
                "round_type": round_data["type"],
                "question": question,
                "answer_length": len(answer.split()),
                "answer_text": answer,
                "is_followup": False
            })

            all_scores.append(analysis)

            if analysis.get("needs_followup"):

                followup = generate_followup(
                    question,
                    answer,
                    analysis["followup_type"]
                )

                print(f"\nAI Follow-up: {followup}\n")

                if voice_mode:
                    text_to_speech(followup)
                    audio_file = record_audio()
                    followup_answer = speech_to_text(audio_file)
                else:
                    followup_answer = input("Your Answer: ")

                recent_answers = [s.get("answer_text", "") for s in all_scores[-5:] if s.get("answer_text")]
                followup_analysis = analyze_answer(
                    topic,
                    current_difficulty,
                    round_data["type"],
                    followup,
                    followup_answer,
                    previous_answers=recent_answers,
                )

                followup_analysis.update({
                    "topic": topic,
                    "difficulty": current_difficulty,
                    "round_type": round_data["type"],
                    "question": followup,
                    "answer_length": len(followup_answer.split()),
                    "answer_text": followup_answer,
                    "is_followup": True
                })

                all_scores.append(followup_analysis)

    return all_scores


def summarize_interview(scores):

    if not scores:
        return {
            "average_depth": 0,
            "average_clarity": 0,
            "average_confidence": 0
        }

    avg_depth = sum(s["depth"] for s in scores) / len(scores)
    avg_clarity = sum(s["clarity"] for s in scores) / len(scores)
    avg_confidence = sum(s["confidence"] for s in scores) / len(scores)

    return {
        "average_depth": round(avg_depth, 2),
        "average_clarity": round(avg_clarity, 2),
        "average_confidence": round(avg_confidence, 2)
    }

# feedback_agent.py

In [16]:
class InterviewFeedback(BaseModel):
    strengths: List[str]
    weaknesses: List[str]
    improvement_plan: List[str]


def _build_deterministic_findings(summary_data):
    strengths = []
    weaknesses = []
    improvement_plan = []

    if summary_data["avg_depth"] >= 75:
        strengths.append("Strong role-relevant depth across most answers.")
    elif summary_data["avg_depth"] < 60:
        weaknesses.append("Depth is below expected level for this role.")
        improvement_plan.append("Use concrete examples with decisions, trade-offs, and outcomes to strengthen depth.")

    if summary_data["avg_clarity"] >= 75:
        strengths.append("Strong clarity and structure in responses.")
    elif summary_data["avg_clarity"] < 60:
        weaknesses.append("Response clarity needs improvement.")
        improvement_plan.append("Use a clear structure (context, action, result) in each answer.")

    if summary_data["avg_confidence"] >= 75:
        strengths.append("Good confidence and ownership in explanations.")
    elif summary_data["avg_confidence"] < 60:
        weaknesses.append("Confidence is inconsistent in key responses.")
        improvement_plan.append("State decisions more directly and reduce hedging language.")

    if summary_data["avg_speech_clarity"] >= 75:
        strengths.append("Speech delivery is generally clear and understandable.")
    elif summary_data["avg_speech_clarity"] < 60:
        weaknesses.append("Speech clarity is below target.")
        improvement_plan.append("Practice concise spoken delivery with fewer filler phrases.")

    if summary_data["followups_triggered"] > 3:
        weaknesses.append("Frequent follow-ups indicate some responses lacked initial completeness.")
        improvement_plan.append("Answer the main question first, then add one concrete example and impact.")

    if not strengths:
        strengths.append("Shows consistent engagement across interview rounds.")

    if not weaknesses:
        weaknesses.append("No major weaknesses detected from current interview metrics.")

    if not improvement_plan:
        improvement_plan.append("Maintain consistency by continuing structured, concise, evidence-backed answers.")

    return strengths, weaknesses, improvement_plan


def _merge_unique(primary, fallback, limit=5):
    merged = []
    for item in (primary or []) + (fallback or []):
        txt = str(item).strip()
        if txt and txt not in merged:
            merged.append(txt)
        if len(merged) >= limit:
            break
    return merged


def generate_feedback(all_scores):

    if not all_scores:
        return {"error": "Interview incomplete. Not enough data."}

    round_scores = defaultdict(list)

    for s in all_scores:
        round_scores[s["round_type"]].append(s["depth"] )

    total_questions = len(all_scores)

    short_answers = sum(1 for s in all_scores if s.get("answer_length", 0) < 20)

    avg_depth = sum(s["depth"] for s in all_scores) / total_questions
    avg_clarity = sum(s["clarity"] for s in all_scores) / total_questions
    avg_conf = sum(s["confidence"] for s in all_scores) / total_questions
    avg_speech = sum(s.get("speech_clarity", 0) for s in all_scores) / total_questions

    weak_depth = sum(1 for s in all_scores if s["depth"] < 50)
    weak_clarity = sum(1 for s in all_scores if s["clarity"] < 50)
    weak_conf = sum(1 for s in all_scores if s["confidence"] < 50)
    weak_speech = sum(1 for s in all_scores if s.get("speech_clarity", 0) < 50)

    followups_triggered = sum(1 for s in all_scores if s.get("needs_followup"))

    summary_data = {
        "total_questions": total_questions,
        "avg_depth": round(avg_depth, 2),
        "avg_clarity": round(avg_clarity, 2),
        "avg_confidence": round(avg_conf, 2),
        "avg_speech_clarity": round(avg_speech, 2),
        "weak_depth_count": weak_depth,
        "weak_clarity_count": weak_clarity,
        "weak_confidence_count": weak_conf,
        "weak_speech_clarity_count": weak_speech,
        "followups_triggered": followups_triggered,
        "short_answers": short_answers
    }

    deterministic_strengths, deterministic_weaknesses, deterministic_plan = _build_deterministic_findings(summary_data)

    system_prompt = """
    You are an interview feedback assistant for universal role interviews.

    Use the provided metrics to produce concise structured feedback.

    Rules:
    - Only mention a weakness if the metric data supports it.
    - Do not claim weak speech clarity when avg_speech_clarity >= 75 and weak_speech_clarity_count <= 1.
    - Do not claim weak confidence when avg_confidence >= 75 and weak_confidence_count <= 1.
    - Keep advice specific, actionable, and conservative.

    RETURN JSON FORMAT ONLY
    """

    user_prompt = f"""
    Performance Metrics:
    {summary_data}

    Generate structured feedback.
    """

    feedback = generate_with_mode(
        mode="feedback",
        system_prompt=system_prompt,
        user_prompt=user_prompt,
        schema_model=InterviewFeedback   # STRICT STRUCTURE
    )

    merged_strengths = _merge_unique(feedback.get("strengths", []), deterministic_strengths, limit=5)
    merged_weaknesses = _merge_unique(feedback.get("weaknesses", []), deterministic_weaknesses, limit=5)
    merged_plan = _merge_unique(feedback.get("improvement_plan", []), deterministic_plan, limit=5)

    if summary_data["avg_speech_clarity"] >= 75 and summary_data["weak_speech_clarity_count"] <= 1:
        merged_weaknesses = [w for w in merged_weaknesses if "speech" not in w.lower() and "spoken" not in w.lower()]

    if summary_data["avg_confidence"] >= 75 and summary_data["weak_confidence_count"] <= 1:
        merged_weaknesses = [w for w in merged_weaknesses if "confidence" not in w.lower()]

    if not merged_weaknesses:
        merged_weaknesses = ["No major weaknesses detected from current interview metrics."]

    return {
        "metrics": summary_data,
        "structured_feedback": {
            "strengths": merged_strengths,
            "weaknesses": merged_weaknesses,
            "improvement_plan": merged_plan,
        }
    }

In [17]:
!apt-get update
!apt-get install -y libportaudio2 libasound-dev

Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]
Get:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:4 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:5 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [85.2 kB]
Get:6 https://cli.github.com/packages stable/main amd64 Packages [357 B]
Hit:7 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:8 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,385 kB]
Get:9 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:10 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:11 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Hit:12 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Get:13 http://security.ubuntu.com/ubuntu jammy

# voice_input.py

In [18]:
import sounddevice as sd
import numpy as np
import scipy.io.wavfile as wav
from faster_whisper import WhisperModel

model = WhisperModel("medium")


def record_audio(duration=10, filename="input.wav"):
    fs = 16000

    input("Press Enter and start speaking...")
    print("Recording...")

    audio = sd.rec(int(duration * fs), samplerate=fs, channels=1)
    sd.wait()

    audio = audio / np.max(np.abs(audio))

    wav.write(filename, fs, audio)
    return filename


def speech_to_text(filename):
    segments, _ = model.transcribe(
        filename, beam_size=5, language="en", vad_filter=True
    )

    text = ""
    for segment in segments:
        text += segment.text
        print(f'[{segment.start:.2f}s - {segment.end:.2f}s]: {segment.text}')

    return text.strip()

In [19]:
!pip install sounddevice soundfile

# voice_output.py

In [20]:
import asyncio
import edge_tts
import sounddevice as sd
import soundfile as sf
import os


async def speak_async(text):
    file_path = "tts_output.mp3"

    communicate = edge_tts.Communicate(text, "en-US-GuyNeural")
    await communicate.save(file_path)

    # Read MP3 directly
    data, samplerate = sf.read(file_path, dtype="float32")

    # Play audio directly in terminal (no external app)
    sd.play(data, samplerate)
    sd.wait()

    os.remove(file_path)


def text_to_speech(text):
    try:
        print("AI says:", text)
        loop = asyncio.get_running_loop()
        loop.create_task(speak_async(text))
    except RuntimeError:
        asyncio.run(speak_async(text))

# terminal_interview.py

In [21]:
def main():

    print("=== AI Mock Interview Platform ===\n")

    resume_path = input("Enter resume file path (.pdf or .txt): ")
    jd_path = input("Enter job description file path (.txt): ")

    print("\nLoading files...")
    resume_text = load_resume(resume_path)
    jd_text = load_resume(jd_path)

    try:
        print("\nParsing resume...")
        resume_data = robust_parse_resume(resume_text)

        print("Parsing job description...")
        jd_data = robust_parse_jd(jd_text)

        print("Analyzing skill gaps...")
        gap_data = analyze_gap(resume_data, jd_data)

        print("\nGenerating interview plan...")
        plan = generate_interview_plan(
            gap_data,
            resume_data.get("experience_years", 0),
        )

    except Exception as e:
        print(f"\nSystem Error During Setup: {e}")
        return

    if hasattr(plan, "model_dump"):
        plan_data = plan.model_dump()
    else:
        plan_data = plan

    # 🧠 Clean Plan Display
    print("\n===== INTERVIEW PLAN =====")
    print(f"Difficulty: {plan_data['difficulty']}\n")

    for i, rnd in enumerate(plan_data["rounds"], 1):
        print(f"Round {i}: {rnd['type']}")
        print("Focus Areas:", ", ".join(rnd["focus_areas"]))
        print("Stress Test Areas:", ", ".join(rnd["stress_test_areas"]))
        print()

    print("\nStarting Interview...\n")

    mode = input("Select mode (1 = Text, 2 = Voice): ")
    voice_mode = True if mode == "2" else False

    scores = run_interview(plan_data, voice_mode=voice_mode)

    print("\nGenerating Final Feedback...\n")
    feedback_data = generate_feedback(scores)

    print("\n===== FINAL REPORT =====\n")

    # 📊 Metrics
    metrics = feedback_data.get("metrics", {})
    structured_feedback = feedback_data.get("structured_feedback", {})

    print("----- PERFORMANCE METRICS -----")
    for key, value in metrics.items():
        print(f"{key}: {value}")

    print("\n----- STRENGTHS -----")
    for item in structured_feedback.get("strengths", []):
        print(f"- {item}")

    print("\n----- WEAKNESSES -----")
    for item in structured_feedback.get("weaknesses", []):
        print(f"- {item}")

    print("\n----- IMPROVEMENT PLAN -----")
    for item in structured_feedback.get("improvement_plan", []):
        print(f"- {item}")

    print("\nInterview Completed.\n")


if __name__ == "__main__":
    main()

=== AI Mock Interview Platform ===

Enter resume file path (.pdf or .txt): /content/SumeetDutta_InternshalaResume.pdf
Enter job description file path (.txt): /content/jd.txt

Loading files...

Parsing resume...
Parsing job description...
Analyzing skill gaps...

Gap Analysis:

strong_skills: ['Node.js', 'Express', 'MongoDB']

partial_match_skills: ['Backend development', 'Data Analytics']

skill_gaps: ['REST APIs', 'JWT Authentication']

resume_experience_years: 1.0

experience_required: 2 years

readiness_score: 43

Generating interview plan...

Interview Plan:

difficulty: BEGINNER

rounds: [{'type': 'Warm-up', 'focus_areas': ['Candidate Introduction', 'Experience Overview', 'Career Aspirations'], 'stress_test_areas': []}, {'type': 'Technical', 'focus_areas': ['Node.js', 'Express'], 'stress_test_areas': []}, {'type': 'Case Study', 'focus_areas': ['REST APIs', 'Backend development'], 'stress_test_areas': ['MongoDB']}, {'type': 'Behavioral', 'focus_areas': ['Data Analytics', 'Backend d